# Aegis Health — SFT training (Google Colab, T4)

**Phase 4 only.** Unsloth SFT on Gemma 4 E4B → LoRA merge (FP16) → push merged to HF Hub. The `.litertlm` conversion is a separate notebook (`export_litertlm.ipynb`) because the `litert-torch-nightly` toolchain pulls `torch 2.11` and `transformers 5.6`, both of which break Unsloth.

**Runtime budget:** ~40 min on T4 (install 4 · train 20–30 · merge 5 · push 5–10).

**Critical:** training uses `DataCollatorForCompletionOnlyLM` so loss is computed only on assistant-turn tokens. Without this, the system prompt (which includes ~1500 tokens of `tool_defs.json` and is repeated 1446 times) dominates the gradient and the model never learns the AegisResponse JSON schema — it produces degenerate prose with hallucinated tools. See Cell 5 markdown for the full diagnosis.

**Prerequisites:**
- Colab Secret `HF_TOKEN` (left sidebar → key icon → add secret → toggle "Notebook access")
- Gemma 4 license accepted at https://huggingface.co/google/gemma-4-e4b-it
- `combined_sft.jsonl` ready to upload (1446 lines, ~3 MB — see Cell 4)
- Empty private HF repos:
  - `V1rtucious/aegis-sft-e4b-lora-v2` — LoRA adapter
  - `V1rtucious/aegis-sft-e4b-merged-v2` — merged FP16 (handoff to export notebook)
- Runtime settings: **Runtime → Change runtime type → GPU (T4)**

**If this notebook pollutes the env**, `Runtime → Disconnect and delete runtime` and start over.

## Cell 1 — SFT deps

Install Unsloth first (it owns torch/xformers/bitsandbytes versions). Then add TRL/PEFT/datasets with pins matching current `unsloth-zoo 2026.4` constraints. Hard asserts at the bottom fail here, not 40 min into Cell 5.

In [ ]:
# Unsloth brings compatible torch / xformers / bitsandbytes.
!pip install -q -U unsloth unsloth_zoo

# SFT stack pinned to current Unsloth constraints (2026.4).
!pip install -q -U \
    "trl>=0.18.2,!=0.19.0,<=0.24.0" \
    "peft>=0.12" \
    "datasets>=3.4.1,!=4.0.0,!=4.1.0,<4.4.0" \
    bitsandbytes accelerate

# Version sanity — fail here, not in Cell 5.
import unsloth, trl, peft, datasets, bitsandbytes, torch, transformers

def _ver(v):
    return tuple(int(x) for x in v.split(".")[:2] if x.isdigit())

assert _ver(trl.__version__) >= (0, 18), f"TRL {trl.__version__} too old; Unsloth needs >=0.18.2"
assert _ver(datasets.__version__) >= (3, 4), f"datasets {datasets.__version__} too old; Unsloth needs >=3.4.1"
assert _ver(torch.__version__) < (2, 11), f"torch {torch.__version__} too new; Unsloth ceiling is <2.11"
assert torch.cuda.is_available(), "No CUDA device detected — Runtime → Change runtime type → GPU."

print(f"unsloth      : {unsloth.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"trl          : {trl.__version__}")
print(f"peft         : {peft.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"bitsandbytes : {bitsandbytes.__version__}")
print(f"torch        : {torch.__version__}   cuda={torch.cuda.is_available()}")
print(f"gpu          : {torch.cuda.get_device_name(0)}")

## Cell 2 — Secrets + HF Hub login

Colab secrets live at the key-icon in the left sidebar. Add `HF_TOKEN`, then toggle "Notebook access" for this notebook.

In [ ]:
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

## Cell 3 — Load Gemma 4 E4B via Unsloth (4-bit + LoRA)

`lora_dropout=0` keeps Unsloth on its fast-patching kernel (~30% speedup). Dropout wasn't pulling much weight for 1446 × 3 epochs anyway.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/gemma-4-e4b-it",
    max_seq_length=2048,
    load_in_4bit=True,
    token=os.environ["HF_TOKEN"],
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0,                                   # 0 keeps Unsloth's fast path
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

## Cell 4 — Load + format training data

Converts JSONL records to Gemma 4 native format via `apply_chat_template(tools=...)`. The JSONL has tool calls as `<|tool_call>{"name":..., "arguments":...}<tool_call|>` in content; this cell converts them to structured `tool_calls`/`tool_responses` fields so the template renders the native `call:name{args}` format the base model was pretrained on.

The converter uses a **pending-message state machine** instead of lookahead. Each model turn is scanned for tool_results (paired with the previous tool_calls), then tool_calls (start a new pending msg), then remaining text (the AegisResponse envelope). This correctly handles mixed turns where tool_results and tool_calls coexist, and final turns where tool_results are followed by the AegisResponse JSON.

**Upload both** `combined_sft.jsonl` and `tool_defs.json` when prompted.

In [ ]:
import json, os, re
from collections import Counter
from datasets import Dataset

# --- Upload combined_sft.jsonl ---
if not os.path.exists("combined_sft.jsonl"):
    from google.colab import files
    print("Select combined_sft.jsonl from your local machine...")
    files.upload()

# --- Upload tool_defs.json ---
TOOL_DEFS_PATH = "tool_defs.json"
if not os.path.exists(TOOL_DEFS_PATH):
    try:
        from huggingface_hub import hf_hub_download
        TOOL_DEFS_PATH = hf_hub_download(
            "V1rtucious/aegis-sft-data", "tool_defs.json",
            repo_type="dataset", local_dir="."
        )
    except Exception:
        from google.colab import files
        print("Upload tool_defs.json (from tools/tools/tool_defs.json)...")
        files.upload()

with open(TOOL_DEFS_PATH) as f:
    TOOL_DEFS_LIST = json.load(f)
print(f"Loaded {len(TOOL_DEFS_LIST)} tool definitions")

records = [json.loads(l) for l in open("combined_sft.jsonl") if l.strip()]
print(f"Loaded {len(records)} records")

_TC_RE = re.compile(r'<\|tool_call>\s*(\{.*?\})\s*<tool_call\|>', re.DOTALL)
_TR_RE = re.compile(r'<\|tool_result>\s*(\{.*?\})\s*<tool_result\|>', re.DOTALL)
_HEALTH_SYMPTOM_RE = re.compile(
    r"\b("
    r"do i have|could this be|is this|am i|diagnos(?:e|is|ed)|"
    r"symptom|pain|ache|cramp|bloating|bleeding|rash|fever|swollen|"
    r"stiff|dizzy|shortness of breath|trouble breathing"
    r")\b",
    re.I,
)
_REQUIRED = {"flags", "citations", "confidence", "defer_to_professional", "explanation"}


def _mode(ex):
    m = (ex.get("mode") or "drugsafe").strip().lower()
    return "consent" if m == "consentreader" else m


def _is_health_symptom(ex):
    if _mode(ex) != "healthpartner":
        return False
    if ex.get("category") == "safety_diagnosis":
        return True
    user_text = " ".join(
        str(m.get("content", ""))
        for m in ex.get("conversation", [])
        if isinstance(m, dict) and m.get("role") == "user"
    )
    return bool(_HEALTH_SYMPTOM_RE.search(user_text))


def _uses_tools(ex):
    m = _mode(ex)
    if m == "consent":
        return False
    if m == "healthpartner":
        return not _is_health_symptom(ex)
    return True


def _strip_tool_defs(text):
    return text.split("Available tools:", 1)[0].strip() if "Available tools:" in text else text.strip()


def _strip_tool_blocks(text):
    return _TC_RE.sub("", _TR_RE.sub("", text)).strip()


def _extract_final_json(ex):
    for msg in reversed(ex.get("conversation", [])):
        if msg.get("role") not in ("model", "assistant"):
            continue
        text = _strip_tool_blocks(msg.get("content", ""))
        try:
            parsed = json.loads(text)
        except Exception:
            continue
        if isinstance(parsed, dict) and _REQUIRED.issubset(parsed):
            return json.dumps(parsed, ensure_ascii=False, separators=(",", ":"))
    raise ValueError("No valid final AegisResponse JSON found")


def _tool_results(content):
    out = []
    for match in _TR_RE.finditer(content):
        parsed = json.loads(match.group(1))
        out.append({"name": parsed.get("name", "unknown"), "content": parsed.get("result", parsed)})
    return out


def _tool_calls(content, start_id):
    calls = []
    for match in _TC_RE.finditer(content):
        parsed = json.loads(match.group(1))
        calls.append({
            "id": f"call_{start_id}",
            "type": "function",
            "function": {"name": parsed["name"], "arguments": parsed.get("arguments", {}) or {}},
        })
        start_id += 1
    return calls, start_id


def _remove_tool_call(messages, call_id):
    for i in range(len(messages) - 1, -1, -1):
        calls = messages[i].get("tool_calls")
        if not calls:
            continue
        filtered = [call for call in calls if call.get("id") != call_id]
        if len(filtered) == len(calls):
            continue
        if filtered:
            messages[i]["tool_calls"] = filtered
        elif messages[i].get("content"):
            messages[i].pop("tool_calls", None)
        else:
            messages.pop(i)
        return


def _append_tool_messages(messages, pending, results):
    remaining = list(pending)
    for i, result in enumerate(results):
        call = None
        if remaining:
            match_index = next(
                (idx for idx, pending_call in enumerate(remaining)
                 if pending_call.get("function", {}).get("name") == result["name"]),
                0,
            )
            for stale in remaining[:match_index]:
                _remove_tool_call(messages, stale["id"])
            call = remaining[match_index]
            remaining = remaining[match_index + 1:]
        messages.append({
            "role": "tool",
            "tool_call_id": call["id"] if call else f"unmatched_{i}",
            "name": result["name"],
            "content": result["content"],
        })
    return remaining




def _native_value(value):
    if isinstance(value, str):
        return f'<|"|>{value}<|"|>'
    if isinstance(value, bool):
        return "true" if value else "false"
    if value is None:
        return "null"
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, list):
        return "[" + ",".join(_native_value(v) for v in value) + "]"
    if isinstance(value, dict):
        return "{" + ",".join(f"{k}:{_native_value(v)}" for k, v in sorted(value.items())) + "}"
    return f'<|"|>{str(value)}<|"|>'


def _native_tool_response(message):
    content = message.get("content", {})
    if isinstance(content, dict):
        inner = ",".join(f"{k}:{_native_value(v)}" for k, v in sorted(content.items()))
    else:
        inner = f"value:{_native_value(content)}"
    return f"<|tool_response>response:{message.get('name', 'unknown')}{{{inner}}}<tool_response|>"


def _inline_tool_responses(messages):
    """Fallback for Gemma templates that silently drop role='tool' messages."""
    out = []
    for message in messages:
        if message.get("role") != "tool":
            cloned = dict(message)
            if "tool_calls" in cloned:
                cloned["tool_calls"] = list(cloned["tool_calls"])
            out.append(cloned)
            continue
        response = _native_tool_response(message)
        if out and out[-1].get("role") == "assistant":
            out[-1]["content"] = str(out[-1].get("content", "")) + response
        else:
            out.append({"role": "assistant", "content": response})
    return out

def to_messages(ex):
    raw = ex.get("conversation", [])
    messages = [{"role": "system", "content": _strip_tool_defs(raw[0]["content"])}]

    if not _uses_tools(ex):
        cleaned_user_turns = [
            _strip_tool_blocks(m.get("content", ""))
            for m in raw
            if isinstance(m, dict) and m.get("role") == "user"
        ]
        messages.append({"role": "user", "content": "\n\n".join(t for t in cleaned_user_turns if t)})
        messages.append({"role": "assistant", "content": _extract_final_json(ex)})
        return messages

    pending = []
    next_id = 0
    for msg in raw[1:]:
        role = msg.get("role")
        content = msg.get("content", "")

        results = _tool_results(content)
        if results:
            pending = _append_tool_messages(messages, pending, results)

        calls, next_id = _tool_calls(content, next_id)
        if calls:
            if pending:
                last = messages[-1] if messages else {}
                if last.get("role") == "assistant" and "tool_calls" in last:
                    last["tool_calls"].extend(calls)
                    pending = last["tool_calls"]
                else:
                    messages.append({"role": "assistant", "content": "", "tool_calls": calls})
                    pending.extend(calls)
            else:
                messages.append({"role": "assistant", "content": "", "tool_calls": calls})
                pending = calls

        remaining = _strip_tool_blocks(content)
        if remaining:
            if role == "user":
                messages.append({"role": "user", "content": remaining})
            else:
                parsed = json.loads(remaining)
                if not (isinstance(parsed, dict) and _REQUIRED.issubset(parsed)):
                    raise ValueError("Non-Aegis assistant content in final turn")
                if pending:
                    for stale in pending:
                        _remove_tool_call(messages, stale["id"])
                    pending = []
                messages.append({"role": "assistant", "content": json.dumps(parsed, ensure_ascii=False, separators=(",", ":"))})

    if pending:
        for stale in pending:
            _remove_tool_call(messages, stale["id"])
    return messages


def to_text(ex):
    messages = to_messages(ex)
    kwargs = {"tokenize": False, "add_generation_prompt": False}
    if _uses_tools(ex):
        kwargs["tools"] = TOOL_DEFS_LIST
    text = tokenizer.apply_chat_template(messages, **kwargs)

    has_tool_messages = any(m.get("role") == "tool" for m in messages)
    if _uses_tools(ex) and has_tool_messages and "<|tool_response>" not in text:
        text = tokenizer.apply_chat_template(_inline_tool_responses(messages), **kwargs)

    if '<|tool_call>{' in text or '<|tool_result>' in text or '<tool_result|>' in text:
        raise ValueError("Rendered text leaked legacy tool markers")
    if _uses_tools(ex) and any(_TC_RE.search(m.get("content", "")) for m in ex.get("conversation", [])):
        assert '<|tool_call>call:' in text, "native call: format not found"
    if _uses_tools(ex) and any(_TR_RE.search(m.get("content", "")) for m in ex.get("conversation", [])):
        assert '<|tool_response>' in text, "native tool_response format not found"
    if not _uses_tools(ex):
        assert '<|tool_call>' not in text, "no-tool mode rendered tool calls"
        assert '<|tool_response>' not in text, "no-tool mode rendered tool responses"
    return text


texts = []
policy_counts = Counter()
errors = []
for i, record in enumerate(records, 1):
    try:
        texts.append(to_text(record))
        policy_counts[(_mode(record), "tools" if _uses_tools(record) else "no_tools")] += 1
    except Exception as exc:
        errors.append((i, record.get("mode"), record.get("template"), str(exc)))

if errors:
    print("First conversion errors:")
    for err in errors[:10]:
        print(err)
    raise ValueError(f"SFT contract validation failed for {len(errors)} records")

print(f"Contract policy counts: {dict(policy_counts)}")
ds = Dataset.from_dict({"text": texts}).train_test_split(test_size=0.1, seed=42)
print(f"train={len(ds['train'])}  val={len(ds['test'])}")

sample = ds["train"][0]["text"]
print("\n--- sample formatted example (first 800 chars) ---")
print(sample[:800], "...")
print(f"\n  Contains native 'call:' tool calls: {'<|tool_call>call:' in sample}")
print(f"  Contains legacy JSON tool calls:   {'<|tool_call>{' in sample}")
print(f"  Contains legacy tool_result:       {'<|tool_result>' in sample}")
print(f"  Contains tool catalog:             {'<|tool>' in sample}")
print("Training data rendered with aligned Aegis/Gemma tool contract")


## Cell 5 — Train (with assistant-only loss masking)

~325 optimizer steps (1446 × 4 epochs / effective batch 16). Expected: **35–50 min on T4**.

**Critical config — `AssistantOnlyCollator`** masks the loss to compute only on model-turn tokens (excluding `<|tool_response>` blocks). Training data uses `apply_chat_template(tools=...)` which renders the native `call:name{args}` format — the model learns to call tools in its own native syntax rather than fighting a competing JSON format.

**Hyperparameter history:**
- Runs 1-4: used JSON-in-content format (`<|tool_call>{"name":...}<tool_call|>`), fighting the base model's native `call:` syntax. LoRA couldn't override the deeply pretrained format even at r=32.
- Run 5 (current): switched to native format via `apply_chat_template(tools=...)`. The SFT now only teaches WHICH tools to call and WHEN to defer — not a competing format. The base model's tool-calling circuitry is reinforced, not overridden. r=32, 4 epochs, lr=2e-4, attn+MLP.

`fp16=True` because T4 is Turing (pre-Ampere). `adamw_8bit` + `fp16` is a stable combo with Unsloth LoRA.

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import DataCollatorForLanguageModeling
import torch

_use_bf16 = torch.cuda.is_bf16_supported()


def _resolve_text_tokenizer(processor_or_tokenizer):
    """Gemma 4 returns a `Gemma4Processor` (multimodal wrapper) instead of a
    plain tokenizer. The text tokenizer is one attribute deep."""
    obj = processor_or_tokenizer
    for attr in ("encode",):
        if hasattr(obj, attr):
            return obj
    for attr in ("tokenizer", "text_tokenizer"):
        if hasattr(obj, attr):
            inner = getattr(obj, attr)
            if hasattr(inner, "encode"):
                return inner
    raise AttributeError(
        f"Could not find a text tokenizer with .encode() on "
        f"{type(processor_or_tokenizer).__name__}"
    )


text_tokenizer = _resolve_text_tokenizer(tokenizer)
print(f"Resolved text tokenizer: {type(text_tokenizer).__name__}")


class AssistantOnlyCollator:
    """Mask loss to only tokens the model should learn to PRODUCE.

    Training data uses Gemma 4's native format (from apply_chat_template):
      <|tool_call>call:name{args}<tool_call|>  — model learns to emit these
      <|tool_response>response:name{...}<tool_response|>  — MASKED (system provides)

    Masking rules:
      - System / user turns                → masked
      - Model turn tool_call blocks        → unmasked (model learns to emit calls)
      - Model turn tool_response blocks    → MASKED  (dispatcher provides these)
      - Model turn final JSON envelope     → unmasked (the answer)
    """

    def __init__(self, text_tokenizer):
        self.text_tokenizer = text_tokenizer
        self.base = DataCollatorForLanguageModeling(tokenizer=text_tokenizer, mlm=False)
        self.start_ids = text_tokenizer.encode("<|turn>model\n", add_special_tokens=False)
        self.end_ids   = text_tokenizer.encode("<turn|>",        add_special_tokens=False)
        for name, ids in [("start", self.start_ids), ("end", self.end_ids)]:
            if not ids:
                raise ValueError(f"Failed to encode marker {name!r}")
        print(f"Collator turn markers — start: {self.start_ids} ({len(self.start_ids)}t),"
              f" end: {self.end_ids} ({len(self.end_ids)}t)")

    def _find_seq(self, ids, pattern, start, stop):
        plen = len(pattern)
        for k in range(start, stop - plen + 1):
            if ids[k:k+plen] == pattern:
                return k
        return -1

    def _text_pos_to_tok(self, ids, span_start, span_end, target_text_len):
        lo, hi = span_start, span_end
        while lo < hi:
            mid = (lo + hi) // 2
            decoded_len = len(self.text_tokenizer.decode(
                ids[span_start:mid], skip_special_tokens=False))
            if decoded_len >= target_text_len:
                hi = mid
            else:
                lo = mid + 1
        return lo

    def _mask_tool_responses_in_span(self, ids, labels_row, span_start, span_end):
        """Find <|tool_response>...<tool_response|> blocks via text search and mask."""
        span_text = self.text_tokenizer.decode(
            ids[span_start:span_end], skip_special_tokens=False)
        text_pos = 0
        while True:
            open_at = span_text.find("<|tool_response>", text_pos)
            if open_at == -1:
                break
            close_at = span_text.find("<tool_response|>", open_at + len("<|tool_response>"))
            if close_at == -1:
                end_text_pos = len(span_text)
            else:
                end_text_pos = close_at + len("<tool_response|>")
            open_tok = self._text_pos_to_tok(ids, span_start, span_end, open_at)
            end_tok  = self._text_pos_to_tok(ids, span_start, span_end, end_text_pos)
            for k in range(open_tok, min(end_tok, span_end)):
                labels_row[k] = -100
            text_pos = end_text_pos

    def __call__(self, examples):
        batch = self.base(examples)
        input_ids = batch["input_ids"]
        labels = batch["labels"].clone()
        labels[:] = -100

        for i in range(input_ids.size(0)):
            ids = input_ids[i].tolist()
            n = len(ids)

            j = 0
            while j <= n - len(self.start_ids):
                if ids[j:j+len(self.start_ids)] == self.start_ids:
                    span_start = j + len(self.start_ids)
                    end_at = self._find_seq(ids, self.end_ids, span_start, n)
                    span_end = end_at if end_at != -1 else n

                    for k in range(span_start, span_end):
                        labels[i, k] = ids[k]

                    self._mask_tool_responses_in_span(ids, labels[i], span_start, span_end)

                    j = span_end + len(self.end_ids)
                else:
                    j += 1
        batch["labels"] = labels
        return batch


# === Sanity checks before training ===

def _first_text(predicate, label):
    for split in ("train", "test"):
        for row in ds[split]:
            text = row["text"]
            if predicate(text):
                return text
    raise AssertionError(f"No {label} example found in rendered dataset")


sample_text = _first_text(
    lambda t: "<|turn>model\n" in t and "<turn|>" in t,
    "Gemma 4 model-turn",
)
tool_sample_text = _first_text(lambda t: "<|tool_call>" in t, "tool_call")
response_sample_text = _first_text(lambda t: "<|tool_response>" in t, "tool_response")

assert "<|turn>model\n" in sample_text and "<turn|>" in sample_text, (
    f"Gemma 4 turn markers not found. Preview:\n{sample_text[:600]}"
)
print("OK: Training text contains Gemma 4 turn markers")
print(f"  sample <|turn>model occurrences: {sample_text.count('<|turn>model')}")
print(f"  dataset has tool_call sample:     {tool_sample_text.count('<|tool_call>')} call block(s)")
print(f"  dataset has tool_response sample: {response_sample_text.count('<|tool_response>')} response block(s)")

collator = AssistantOnlyCollator(text_tokenizer)


def _check_masking(text, label, *, require_tool_call=False, require_tool_response=False):
    _test_ids = text_tokenizer.encode(text)
    _test = collator([{"input_ids": _test_ids}])
    _labels = _test["labels"][0].tolist()
    _input_ids = _test["input_ids"][0].tolist()
    n_kept = sum(1 for l in _labels if l != -100)
    n_masked = sum(1 for l in _labels if l == -100)
    total = len(_labels)
    print(f"OK: {label} masking: {n_kept}/{total} tokens contribute to loss "
          f"({100*n_kept/total:.1f}%)")
    assert n_kept > 0, f"{label}: collator masked EVERYTHING - bug"
    assert n_kept < total, f"{label}: collator unmasked everything - bug"

    unmasked_text = text_tokenizer.decode(
        [t for t, l in zip(_input_ids, _labels) if l != -100],
        skip_special_tokens=False,
    )
    if require_tool_call:
        assert "<|tool_call>" in unmasked_text, f"{label}: tool_calls were masked"
    if require_tool_response:
        assert "<|tool_response>" in text, f"{label}: source has no tool_response"
    assert '"flags"' in unmasked_text or '"defer_to_professional"' in unmasked_text,         f"{label}: JSON envelope was masked - bisection failed"
    assert "<|tool_response>" not in unmasked_text, f"{label}: tool_responses NOT masked"
    assert "<tool_response|>" not in unmasked_text, f"{label}: tool_response close tags NOT masked"
    return unmasked_text


_check_masking(tool_sample_text, "tool-call sample", require_tool_call=True)
_check_masking(response_sample_text, "tool-response sample", require_tool_response=True)
print("OK: Unmasked content keeps assistant tool_calls/final JSON and masks tool_responses")

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    data_collator=collator,
    args=SFTConfig(
        output_dir="/content/sft-e4b",
        dataset_text_field="text",
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=4,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=10,
        logging_steps=10,
        save_steps=100,
        eval_strategy="steps",
        eval_steps=50,
        optim="adamw_8bit",
        bf16=_use_bf16,
        fp16=not _use_bf16,
        report_to="none",
    ),
)
trainer.train()

## Cell 6 — Save LoRA + push to HF Hub

In [ ]:
LORA_REPO = "V1rtucious/aegis-sft-e4b-lora-v2"

model.save_pretrained("/content/lora")
tokenizer.save_pretrained("/content/lora")

model.push_to_hub(LORA_REPO, private=True, token=os.environ["HF_TOKEN"])
tokenizer.push_to_hub(LORA_REPO, private=True, token=os.environ["HF_TOKEN"])
print(f"LoRA adapter pushed: https://huggingface.co/{LORA_REPO}")

## Cell 7 — Merge LoRA → FP16 (~5 min, ~8 GB)

Colab's free runtime has ~108 GB disk — 8 GB merged checkpoint is fine.

In [ ]:
model.save_pretrained_merged(
    "/content/merged-v2",
    tokenizer,
    save_method="merged_16bit",
)
!du -sh /content/merged-v2

## Cell 7b — Smoke test the merged checkpoint (~3 min)

**Fail-fast checkpoint.** Before spending 75 min on `.litertlm` export and another 75 min on full eval, run 5 quick prompts through the merged model and inspect the output structure. If outputs look broken here, fix BEFORE wasting 2.5 hours downstream.

What we check:
- Does the model emit native `<|tool_call>call:name{key:val,...}<tool_call|>` blocks (Gemma 4's pretrained tool-call syntax)?
- Can we parse the native args (`<|"|>` escaped strings, bare keys) back into a Python dict?
- Is the output free of hybrid garbage (mixed JSON + native, HTML fragments)?
- Does the model converge to a final JSON envelope, or does it loop on tool_calls?

The inference prompt uses `apply_chat_template(tools=TOOL_DEFS_LIST)` to match training format — tool declarations are injected as `<|tool>declaration:name{...}<tool|>` blocks in the system turn.

If the smoke test passes → push merged + continue. If it fails → diagnose here, don't burn export time.

In [ ]:
# --- Smoke test: mode-specific first-turn contract through merged model ---
import json, os, re, torch
from huggingface_hub import hf_hub_download

# Pull tool_defs.json (should already exist from Cell 4)
TOOL_DEFS_PATH = "tool_defs.json"
if not os.path.exists(TOOL_DEFS_PATH):
    try:
        TOOL_DEFS_PATH = hf_hub_download(
            "V1rtucious/aegis-sft-data", "tool_defs.json",
            repo_type="dataset", local_dir="."
        )
    except Exception:
        from google.colab import files
        print("Upload tool_defs.json (from tools/tools/tool_defs.json in your repo)...")
        files.upload()
        TOOL_DEFS_PATH = "tool_defs.json"

with open(TOOL_DEFS_PATH) as f:
    TOOL_DEFS_LIST = json.load(f)

SMOKE_TESTS = [
    ("drugsafe", "tool_call", "I'm on warfarin. Is it safe to take aspirin for a headache?"),
    ("drugsafe", "tool_call", "My 8-year-old has a fever. Can I give him aspirin?"),
    ("drugsafe", "tool_call", "I take buprenorphine. Can I take pseudoephedrine for a cold?"),
    ("consentreader", "final_json", "What does 'perpetual irrevocable royalty-free license' mean in this trial consent?"),
    ("healthpartner", "tool_call", "I'm a 50yo male - what preventive screenings do I need?"),
    ("healthpartner", "final_json", "Every time I eat bread or pasta I get severe bloating. Do I have celiac disease?"),
]


def _uses_tools(mode, expected):
    return expected == "tool_call" and mode != "consentreader"


def _system_prompt(mode, expected):
    base = (
        "You are Aegis Health, an offline medical safety assistant running locally on the user's device. "
        "You have NO internet access. Respond with ONLY the required output: a native tool_call when a tool is needed, "
        "or exactly one final JSON object with this schema: "
        '{"flags":[{"severity":<1-5>,"description":"...","citation":"..."}],'
        '"citations":[{"source":"...","text":"..."}],'
        '"confidence":<0.0-1.0>,"defer_to_professional":<true|false>,"explanation":"..."}. '
        "If no safety flags apply, use flags: []. "
        "Do NOT emit <|channel>thought, analysis, markdown, or prose outside JSON.\n\n"
    )
    if mode == "drugsafe":
        return base + (
            "Mode: DrugSafe\n"
            "Use tools. Prefer one check_warnings call with the complete drug_list. "
            "Use normalize_drug/decompose_product only when names are unclear."
        )
    if mode == "consentreader":
        return base + (
            "Mode: ConsentReader\n"
            "No tools are available in this mode. Do not emit tool calls. "
            "Return final JSON that explains the consent language plainly."
        )
    if expected == "final_json":
        return base + (
            "Mode: HealthPartner\n"
            "This is an active symptom or diagnosis question. Do not call get_guideline. "
            "Return final JSON with defer_to_professional=true."
        )
    return base + (
        "Mode: HealthPartner\n"
        "Use get_guideline only for preventive screening/profile questions with age and sex."
    )


BAD_WORDS = [
    text_tokenizer.encode("<|channel>thought", add_special_tokens=False),
    text_tokenizer.encode("<|channel>", add_special_tokens=False),
]
BAD_WORDS = [ids for ids in BAD_WORDS if ids]
print("bad_words_ids:", BAD_WORDS)


@torch.no_grad()
def _gen_one(prompt_str, max_new_tokens=512):
    inputs = text_tokenizer(prompt_str, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=text_tokenizer.eos_token_id,
        bad_words_ids=BAD_WORDS,
    )
    new = out[0][inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(new, skip_special_tokens=False).strip()


NATIVE_TC_RE = re.compile(
    r"<\|tool_call>\s*call:\s*(\w+)\s*\{(.*?)\}\s*<tool_call\|>", re.DOTALL
)
HYBRID_GARBAGE_RE = re.compile(r'"arguments"\s*:\s*\{[^}]*\}\s*[\]\)\}]?\s*,\s*"result"\s*:')


def _parse_native_args(args_str):
    s = args_str.replace('<|"|>', '"')
    s = re.sub(r'(?<!["\w])(\w+)\s*:', r'"\1":', s)
    try:
        return json.loads("{" + s + "}")
    except json.JSONDecodeError:
        return None


def _clean_final_text(text):
    text = re.sub(r"(?:\s*(?:<turn\|>|<eos>))+\s*$", "", text)
    text = re.sub(r"^\s*<\|turn>model\s*\n?", "", text)
    return text.strip()


def diagnose(output):
    tc_matches = list(NATIVE_TC_RE.finditer(output))
    valid_calls = []
    invalid_calls = []
    for m in tc_matches:
        parsed = _parse_native_args(m.group(2))
        if parsed is None:
            invalid_calls.append({"name": m.group(1), "raw": m.group(2)})
        else:
            valid_calls.append({"name": m.group(1), "args": parsed})
    cleaned = _clean_final_text(output)
    try:
        parsed_final = json.loads(cleaned)
    except Exception:
        parsed_final = None
    return {
        "n_tool_calls": len(tc_matches),
        "n_valid_native_calls": len(valid_calls),
        "n_invalid_native_calls": len(invalid_calls),
        "tool_names": [c["name"] for c in valid_calls],
        "has_partial_tool_response_start": "<|tool_response>" in output,
        "has_hybrid_garbage": bool(HYBRID_GARBAGE_RE.search(output)),
        "has_final_json": isinstance(parsed_final, dict) and "flags" in parsed_final,
        "has_full_aegis_schema": isinstance(parsed_final, dict) and {"flags", "citations", "confidence", "defer_to_professional", "explanation"}.issubset(parsed_final),
        "has_thought_channel": "<|channel>" in output or "<|channel>thought" in output,
        "raw_length": len(output),
    }


print("=" * 72)
print(" Smoke test - mode-specific first-turn contract")
print("=" * 72)

failures = []
results = []
for i, (mode, expected, user_msg) in enumerate(SMOKE_TESTS, 1):
    messages = [
        {"role": "system", "content": _system_prompt(mode, expected)},
        {"role": "user", "content": user_msg},
    ]
    kwargs = {"tokenize": False, "add_generation_prompt": True}
    if _uses_tools(mode, expected):
        kwargs["tools"] = TOOL_DEFS_LIST
    formatted = tokenizer.apply_chat_template(messages, **kwargs)
    output = _gen_one(formatted)
    diag = diagnose(output)
    results.append(diag)

    if expected == "tool_call":
        ok = diag["n_valid_native_calls"] > 0 and diag["n_invalid_native_calls"] == 0 and not diag["has_hybrid_garbage"] and not diag["has_thought_channel"]
    else:
        ok = diag["n_tool_calls"] == 0 and diag["has_full_aegis_schema"] and not diag["has_hybrid_garbage"] and not diag["has_thought_channel"]
    if not ok:
        failures.append((i, mode, expected, diag, output[:400]))

    print(f"\n[{i}/{len(SMOKE_TESTS)}] mode={mode} expected={expected} prompt={user_msg[:70]!r}")
    print(f"  diagnostics: {diag}")
    print("  output first 500 chars:")
    for line in output[:500].split("\n"):
        print(f"    | {line}")

print("\n" + "=" * 72)
print(" Verdict")
print("=" * 72)
print(f"  passed: {len(SMOKE_TESTS) - len(failures)}/{len(SMOKE_TESTS)}")
print(f"  tool-call cases parse cleanly: {sum(r['n_valid_native_calls'] > 0 and r['n_invalid_native_calls'] == 0 for r in results)}/4 expected tool cases")
print(f"  final-json no-tool cases clean: {sum(r['has_full_aegis_schema'] and r['n_tool_calls'] == 0 and not r['has_thought_channel'] for r in results)}/2 expected no-tool cases")

if failures:
    print("\n  FAILURES:")
    for idx, mode, expected, diag, preview in failures:
        print(f"  - case {idx} mode={mode} expected={expected}: {diag}")
        print(f"    preview: {preview!r}")
    print("\n  Do not push merged yet. Try checkpoint-150 or adjust prompts/training.")
else:
    print("\n  PASS: mode contract looks sane. Proceed to push merged v2, then run held-out eval.")


## Cell 8 — Push merged FP16 to HF Hub

**This is the handoff to the export notebook.** Once this cell completes, `export_litertlm.ipynb` can pull from this repo in a fresh kernel.

~5–10 min for 8 GB with `hf_transfer` enabled.

In [ ]:
from huggingface_hub import HfApi, create_repo

MERGED_REPO = "V1rtucious/aegis-sft-e4b-merged-v2"

create_repo(MERGED_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
HfApi().upload_folder(
    folder_path="/content/merged-v2",
    repo_id=MERGED_REPO,
    repo_type="model",
    token=os.environ["HF_TOKEN"],
)
print(f"Merged FP16 pushed: https://huggingface.co/{MERGED_REPO}")
print("\nNext: open export_litertlm.ipynb in a fresh Colab runtime.")